In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):

    for filename in filenames:

        print(os.path.join(dirname, filename))
 
import kagglehub

In [ ]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('/kaggle/input/datasets/prarthanap374/world-happiness/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']

In [ ]:
print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])

In [ ]:
# Task 1: Regional comparison bar chart
# -------------------------------------
import plotly.graph_objects as go

# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # ascending = highest bar ends up on top

print(region_avg)

# Step 2: Build the chart
global_avg = df['Happiness_Score'].mean()
top_region = region_avg.iloc[-1]       # highest score (last row after ascending sort)
bottom_region = region_avg.iloc[0]     # lowest score
gap = top_region['Happiness_Score'] - bottom_region['Happiness_Score']

# Design choice 1: highlight the top region, mute everything else
colors = ['#1f77b4' if r == top_region['Region'] else '#c7c7c7'
          for r in region_avg['Region']]

fig = go.Figure(go.Bar(
    x=region_avg['Happiness_Score'],
    y=region_avg['Region'],
    orientation='h',
    marker_color=colors,
    # Design choice 2: direct value labels instead of relying on hover
    text=region_avg['Happiness_Score'].round(2),
    textposition='outside',
    hovertemplate='%{y}: %{x:.2f}<extra></extra>'
))

# Design choice 3: reference line for the global average
fig.add_vline(
    x=global_avg,
    line_dash='dash',
    line_color='#555',
    annotation_text=f'Global avg: {global_avg:.2f}',
    annotation_position='top'
)

fig.update_layout(
    title=dict(
        text=(f"{top_region['Region']} is the happiest region, "
              f"scoring {gap:.1f} points above {bottom_region['Region']}"),
        x=0, xanchor='left'
    ),
    xaxis=dict(
        title='Average happiness score',
        range=[0, region_avg['Happiness_Score'].max() * 1.12],  # zero baseline + room for labels
        showgrid=False
    ),
    yaxis=dict(title=None),
    plot_bgcolor='white',
    showlegend=False,
    margin=dict(l=20, r=40, t=80, b=50),
    height=500
)

fig.show()

In [ ]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------
import plotly.graph_objects as go

# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

# Step 2: Build the chart
top_mean = top8['Happiness_Score'].mean()
bottom_mean = bottom8['Happiness_Score'].mean()
ratio = top_mean / bottom_mean
gap = top_mean - bottom_mean

# Colour by group: two-colour palette instead of the default rainbow
color_map = {'Top 8': '#1f77b4', 'Bottom 8': '#d62728'}
colors = combined['Group'].map(color_map)

fig = go.Figure(go.Bar(
    x=combined['Happiness_Score'],
    y=combined['Country'],
    orientation='h',
    marker_color=colors,
    text=combined['Happiness_Score'].round(2),
    textposition='outside',
    hovertemplate='%{y}: %{x:.2f}<extra></extra>',
    showlegend=False
))

# Legend proxies so the colours are explained
for group, color in color_map.items():
    fig.add_trace(go.Bar(x=[None], y=[None], orientation='h',
                         marker_color=color, name=group))

# Visual separator: dotted line between the two groups (bottom 8 occupy positions 0-7)
fig.add_hline(y=7.5, line_dash='dot', line_color='#888', line_width=1.5)

# Annotation emphasising the gap
fig.add_annotation(
    x=combined['Happiness_Score'].max() * 0.55,
    y=7.5,
    text=f"<b>Gap: {gap:.1f} points</b><br>Top 8 average {top_mean:.2f} vs. {bottom_mean:.2f}",
    showarrow=False,
    bgcolor='white',
    bordercolor='#888',
    borderwidth=1,
    font=dict(size=13)
)

# Stretch goal: global average reference line
fig.add_vline(
    x=global_avg,
    line_dash='dash',
    line_color='#555',
    annotation_text=f'Global avg: {global_avg:.2f}',
    annotation_position='top'
)

fig.update_layout(
    title=dict(
        text=(f"The happiest countries score {ratio:.1f}x higher "
              f"than the least happy, a {gap:.1f}-point gap"),
        x=0, xanchor='left'
    ),
    xaxis=dict(
        title='Happiness score',
        range=[0, combined['Happiness_Score'].max() * 1.12],  # zero baseline
        showgrid=False
    ),
    yaxis=dict(title=None, categoryorder='array',
               categoryarray=combined['Country']),
    plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(l=20, r=40, t=90, b=50),
    height=650
)

fig.show()